# FormosaNLU — Colab portability run

This notebook wraps the same `src.training.train` entry used on the local RTX 4090. It runs one `real_only`, seed-42 QLoRA experiment for portability evidence; it is not the primary M9 compute path.

**Requirements:** a GPU with at least 22.5 GiB usable memory, the `HF_TOKEN` Colab Secret, and `formosanlu_colab_bundle.zip` in the Drive folder configured below. A T4 (16 GB) is intentionally rejected by preflight.

In [ ]:
from pathlib import Path

from google.colab import drive, userdata

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/sdg-portfolio/FormosaNLU-Synth')
BUNDLE = DRIVE_ROOT / 'formosanlu_colab_bundle.zip'
WORKDIR = Path('/content/formosanlu')
GROUP = 'real_only'
SEED = 42
LOCAL_RUN = WORKDIR / 'runs' / 'colab' / GROUP / f'seed_{SEED}'
DRIVE_RUN = DRIVE_ROOT / 'runs' / GROUP / f'seed_{SEED}'
assert BUNDLE.is_file(), f'Missing bundle: {BUNDLE}'
DRIVE_RUN.mkdir(parents=True, exist_ok=True)
print('Bundle:', BUNDLE)
print('Drive output:', DRIVE_RUN)


In [ ]:
import shutil
import subprocess

gpu_csv = subprocess.check_output([
    'nvidia-smi', '--query-gpu=name,memory.total',
    '--format=csv,noheader,nounits'
], text=True).strip()
gpu_name, gpu_memory = [part.strip() for part in gpu_csv.split(',', maxsplit=1)]
gpu_memory_mib = int(gpu_memory)
free_gib = shutil.disk_usage('/content').free / (1024 ** 3)
assert gpu_memory_mib >= 22500, (
    f'{gpu_name} exposes only {gpu_memory_mib} MiB; use an L4-class 24 GB '
    'GPU or an A100. T4 is insufficient.'
)
assert free_gib >= 35, f'Need at least 35 GiB free under /content; got {free_gib:.1f}'
hf_token = userdata.get('HF_TOKEN')
assert hf_token, 'Add HF_TOKEN under Colab Secrets and enable notebook access.'
print(f'GPU preflight passed: {gpu_name}, {gpu_memory_mib} MiB; disk {free_gib:.1f} GiB free')


In [ ]:
import json
import os
import zipfile

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
WORKDIR.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE) as archive:
    archive.extractall(WORKDIR)
bundle_manifest = json.loads((WORKDIR / 'COLAB_BUNDLE_MANIFEST.json').read_text())
source_commit = bundle_manifest['source_commit']
os.environ['FORMOSANLU_SOURCE_COMMIT'] = source_commit
print('Source commit:', source_commit)
print('Bundle files:', len(bundle_manifest['files']))


In [ ]:
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'sync', '--frozen', '--no-dev'], cwd=WORKDIR, check=True)
VENV_PYTHON = WORKDIR / '.venv' / 'bin' / 'python'
version_check = 'import torch, transformers, trl, peft; print(torch.__version__)'
subprocess.run([str(VENV_PYTHON), '-c', version_check], check=True)


In [ ]:
model_script = r'''
import os
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='google/gemma-4-E4B-it',
    local_dir='data/models/gemma-4-E4B-it',
    token=os.environ['HF_TOKEN'],
    allow_patterns=[
        'config.json', 'generation_config.json', 'chat_template.jinja',
        'processor_config.json', 'tokenizer.json', 'tokenizer_config.json',
        'model.safetensors'
    ],
)
'''
download_env = os.environ.copy()
download_env['HF_TOKEN'] = hf_token
subprocess.run([str(VENV_PYTHON), '-c', model_script], cwd=WORKDIR, env=download_env, check=True)
weight = WORKDIR / 'data/models/gemma-4-E4B-it/model.safetensors'
assert weight.stat().st_size > 15_000_000_000, 'Gemma weight download is incomplete'
print(f'Model ready: {weight.stat().st_size / (1024 ** 3):.2f} GiB')


In [ ]:
if any(DRIVE_RUN.iterdir()):
    shutil.copytree(DRIVE_RUN, LOCAL_RUN, dirs_exist_ok=True)
    print('Restored existing Drive checkpoints for resume.')
else:
    LOCAL_RUN.mkdir(parents=True, exist_ok=True)
    print('Starting a fresh Colab portability run.')


In [ ]:
import threading

stop_sync = threading.Event()

def sync_to_drive():
    if not LOCAL_RUN.exists():
        return
    DRIVE_RUN.mkdir(parents=True, exist_ok=True)
    for source in LOCAL_RUN.iterdir():
        target = DRIVE_RUN / source.name
        if source.name.startswith('checkpoint-'):
            if not (source / 'trainer_state.json').is_file():
                continue
            temporary = DRIVE_RUN / f'{source.name}.syncing'
            if temporary.exists():
                shutil.rmtree(temporary)
            shutil.copytree(source, temporary)
            if target.exists():
                shutil.rmtree(target)
            temporary.rename(target)
        elif source.is_file():
            shutil.copy2(source, target)
        elif source.name == 'adapter':
            shutil.copytree(source, target, dirs_exist_ok=True)

def periodic_sync():
    while not stop_sync.wait(120):
        sync_to_drive()
        print('Checkpoint synced to Drive.')

sync_thread = threading.Thread(target=periodic_sync, daemon=True)
sync_thread.start()
train_env = os.environ.copy()
train_env['FORMOSANLU_SOURCE_COMMIT'] = source_commit
command = [
    str(VENV_PYTHON), '-m', 'src.training.train',
    '--group', GROUP,
    '--seed', str(SEED),
    '--output-dir', str(LOCAL_RUN),
    '--resume',
]
try:
    completed = subprocess.run(command, cwd=WORKDIR, env=train_env, check=False)
finally:
    stop_sync.set()
    sync_thread.join(timeout=5)
    sync_to_drive()
assert completed.returncode == 0, f'Training failed with exit code {completed.returncode}'


In [ ]:
run_report = json.loads((LOCAL_RUN / 'run_report.json').read_text())
required = [
    LOCAL_RUN / 'adapter',
    LOCAL_RUN / 'metrics.jsonl',
    LOCAL_RUN / 'env.json',
    LOCAL_RUN / 'config.snapshot.yaml',
    LOCAL_RUN / 'run_report.json',
]
assert run_report['status'] == 'completed'
assert run_report['group'] == GROUP and run_report['seed'] == SEED
assert all(path.exists() for path in required), required
sync_to_drive()
print(json.dumps(run_report, indent=2, ensure_ascii=False))
print('PORTABILITY RUN PASSED; final artifacts synced to', DRIVE_RUN)


## Return to the local project

Download the Drive directory `runs/real_only/seed_42/` as a ZIP and tell Codex where it landed under `C:\Users\3Hml\Downloads`. The local project will place it under `results/colab/real_only/seed_42/` and compare configuration plus validation traces. Do not publish the adapter from Colab.